
Load and Display Data

In [25]:
import pandas as pd
import os
from collections import Counter
import re
from nltk.corpus import stopwords
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.model_selection import RandomizedSearchCV
from pprint import pprint

In [2]:
# Load the datasets
try:
    train_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\raw\training_data_womensUK.csv', encoding='latin1', on_bad_lines='skip')
    test_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\raw\test_data_womensUK.csv', encoding='latin1', on_bad_lines='skip')

    print("Training data loaded successfully.")
    print("Test data loaded successfully.")

except FileNotFoundError as e:
    print(e)
    print("\n Please make sure the files 'training_data_womensUK.csv' and 'test_data_womensUK.csv' are uploaded.")

# Display the first few rows of the training data
print("\nTraining Data Head:")
display(train_df.head())

# Display the first few rows of the test data
print("\nTest Data Head:")
display(test_df.head())

# Display information about the training data
print("\nTraining Data Info:")
train_df.info()

# Display information about the test data
print("\nTest Data Info:")
test_df.info()

Training data loaded successfully.
Test data loaded successfully.

Training Data Head:


,Product ID,Name,Retailer,Brand,Segment,Gender,Category,Color,Activewear,Pattern,...,Current Discount Percentage,Date First Discounted,SKUs Available,Num Replenishments,Days to Majority SKU sellout,Days to First sellout,Season,Description,Care information,Sizes
0,dc10ba7115fc8e45c03ab71a7e8fb67a24546d1faf874d...,Black Lotty Sliders,TK Maxx (UK),Replay,Mass,Women,Footwear,Black,Non-activewear,Product has no pattern data,...,0.0%,NaN,1/1,0,NaN,NaN,ss25,Sliders\nBlack\nLotty style\nReptile effect fr...,Upper: Synthetic\nInner: Synthetic\nSole: Synt...,4
1,46a57ebe0ad93bcf2c613ec304c31fc16f73f6b3d41081...,Kensey' Casual Lightweight Trainers,Debenhams (UK),Moshulu,Mass,Women,Footwear,Pink,Non-activewear,Product has no pattern data,...,0.0%,NaN,6/7,0,NaN,NaN,aw25,Kensey makes every step comfier! In smooth lea...,"Upper: Nubuck (Indigo, Light Green) or Leather...","3, 4, 5, 6, 6.5, 7, 8"
2,e05afa3109a21743ebd9f7ac0fbb1ce218caa82f241e94...,Kalisa Bow Top in Textured Pink,Motel (UK),Motel,Mass,Women,Tops,Pink,Non-activewear,Plain,...,0.0%,3 Jul 2025,3/7,0,173.0,NaN,ss25,The Kalisa top\nin a pink textured material\nF...,NaN,"XXS, XS, S, M, L, XL, XXL"
3,03b218b35f7afc4d29514f2133af648e33dcbd7d93e557...,2000-2015 Gg Canvas Bamboo Handbag,Farfetch (UK),Gucci Pre-Owned,Luxury,Women,Accessories,Brown,Non-activewear,Tile,...,0.0%,15 Jun 2025,1/1,0,NaN,NaN,ss25,Pre-Owned\n2000-2015 GG Canvas Bamboo handbag\...,Outer:\nCanvas 100%\nCanvas,One Size
4,a327ffcf373d56860d2f95ee58c33256ae3516d087bf2e...,Bubble Split Midaxi Skirt,Boohoo (UK),Boohoo,Value,Women,Bottoms,Grey,Non-activewear,Plain,...,50.0%,12 Jul 2024,5/6,0,NaN,NaN,aw24,Sleek midi skirt with a daring side slit\nHigh...,"100% Polyester, Embroidery 100% Viscose","6, 8, 10, 12, 14, 16"



Test Data Head:


,Product ID,Name,Retailer,Brand,Segment,Gender,Category,Color,Activewear,Pattern,...,Current Discount Percentage,Date First Discounted,SKUs Available,Num Replenishments,Days to Majority SKU sellout,Days to First sellout,Season,Description,Care information,Sizes
0,1fd240b318ed3880dc4e15b9212f6410a01b06ac567462...,Yours Curve Black Linen Button Through Midaxi ...,SuperDrug (UK),Super Drug,Value,Women,Dresses,Black,Non-activewear,Plain,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,20
1,5385cc61e6e5a1ea61d8f2ad556e0b2a2647b8ae5666e9...,PixieGirl Petite Pink Jacquard Side Stripe Wid...,SuperDrug (UK),Super Drug,Value,Women,Bottoms,Pink,Non-activewear,Stripes,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,10
2,52d69007a22e64be803de380da0fc0dc2fd2ce51216707...,Yours Curve Black Tropical Floral Print Croppe...,SuperDrug (UK),Super Drug,Value,Women,Bottoms,Multicolour,Non-activewear,Floral,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,16
3,e03515940ce7118dec7d2c9dc7fabde197596434d4aa50...,Blue Vanilla Brown Abstract Cow-print Shirt,SuperDrug (UK),Super Drug,Value,Women,Tops,Brown,Non-activewear,Abstract,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,Change up your leopard print this season with ...,"Main: 80% Viscose, 20% Polyester",One Size
4,89c955b901bd4dddbb77dcc55102274cc0fdc259fbc3e3...,Where's That From Silver Dream Strappy Flat Sa...,SuperDrug (UK),Super Drug,Value,Women,Footwear,Silver,Non-activewear,Product has no pattern data,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,"Brush the shoes with a brush to remove dust, d...",Faux Leather,UK3



Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Product ID                      60000 non-null  object 
 1   Name                            60000 non-null  object 
 2   Retailer                        60000 non-null  object 
 3   Brand                           60000 non-null  object 
 4   Segment                         60000 non-null  object 
 5   Gender                          60000 non-null  object 
 6   Category                        60000 non-null  object 
 7   Color                           60000 non-null  object 
 8   Activewear                      60000 non-null  object 
 9   Pattern                         60000 non-null  object 
 10  Full Price ($)                  60000 non-null  object 
 11  Original Currency               60000 non-null  object 
 12  Full Price 

Clean Data

In [3]:
# Clean the data
def clean_data(df):
    # Strip leading/trailing spaces from column names
    df.columns = df.columns.str.strip()

    # Fill missing values in categorical columns with the mode
    for col in ['Segment', 'Gender', 'Category', 'Color', 'Activewear', 'Pattern', 'Season']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # Fill missing values in numerical columns with the mean
    for col in ['Num Replenishments', 'Days to Majority SKU sellout', 'Days to First sellout']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].mean())

    # Clean and convert currency and percentage columns
    for col in ['Full Price ($)', 'Full Price (original currency)']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')
            df[col] = df[col].fillna(df[col].mean())

    if 'Current Discount Percentage' in df.columns:
        df['Current Discount Percentage'] = pd.to_numeric(df['Current Discount Percentage'].astype(str).str.replace('%', ''), errors='coerce')
        df['Current Discount Percentage'] = df['Current Discount Percentage'].fillna(df['Current Discount Percentage'].mean())


    # Convert 'Date First Discounted' to datetime
    if 'Date First Discounted' in df.columns:
        df['Date First Discounted'] = pd.to_datetime(df['Date First Discounted'], errors='coerce')

    # Fill missing 'Name', 'Description', 'Care information' and 'Sizes'
    for col in ['Name', 'Description', 'Care information', 'Sizes']:
        if col in df.columns:
            df[col] = df[col].fillna('Not available')


    return df

train_df = clean_data(train_df.copy())
test_df = clean_data(test_df.copy())

print("Data cleaning complete.")

Data cleaning complete.


In [4]:
# Combine the 'Name' and 'Description' columns
text_data = ' '.join(train_df['Name'].fillna('') + ' ' + train_df['Description'].fillna('')  + ' ' + train_df['Care information'].fillna(''))

# Tokenize the text data
words = re.findall(r'\w+', text_data.lower())

# Count the frequency of each word
word_counts = Counter(words)

# Display the most common words
print("Most common words:")
display(word_counts.most_common(20))

Most common words:


[('a', 109526),
 ('and', 100121),
 ('the', 99848),
 ('with', 74284),
 ('for', 58135),
 ('to', 52210),
 ('in', 45235),
 ('of', 38019),
 ('this', 35190),
 ('100', 34748),
 ('is', 31748),
 ('not', 29347),
 ('your', 28298),
 ('cotton', 26456),
 ('leather', 24886),
 ('from', 24721),
 ('fit', 24236),
 ('polyester', 23493),
 ('s', 22308),
 ('on', 22128)]

In [5]:
nltk.download('stopwords')

# Identify common words
text_data = ' '.join(train_df['Name'].fillna('') + ' ' + train_df['Description'].fillna(''))
words = re.findall(r'\w+', text_data.lower())
stop_words = set(stopwords.words('english'))
words = [word for word in words if not word in stop_words]
word_counts = Counter(words)
print("Most common words (after removing stop words):")
display(word_counts.most_common(20))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Most common words (after removing stop words):


[('fit', 21525),
 ('dress', 17945),
 ('length', 17838),
 ('style', 17037),
 ('size', 15386),
 ('design', 14172),
 ('neck', 13355),
 ('top', 12443),
 ('perfect', 12405),
 ('fabric', 11326),
 ('â', 11250),
 ('fastening', 11123),
 ('made', 10363),
 ('long', 10168),
 ('front', 10007),
 ('5', 9738),
 ('look', 9691),
 ('sleeve', 9509),
 ('sleeves', 9494),
 ('0', 9257)]

In [6]:
# Combine the text columns
train_df['text'] = train_df['Name'].fillna('') + ' ' + train_df['Description'].fillna('') + ' ' + train_df['Care information'].fillna('')

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=100)

# Fit and transform the combined text data
X_train_text = vectorizer.fit_transform(train_df['text'])

# Convert the transformed data to a pandas DataFrame
X_train_text = pd.DataFrame(X_train_text.toarray(), columns=vectorizer.get_feature_names_out())

# Display the first few rows of the vectorized data
display(X_train_text.head())

,100,adjustable,available,bag,black,bleach,brand,button,care,casual,...,use,versatile,viscose,waist,wash,washable,wear,white,wide,zip
0,0.000000,0.0,0.000000,0.0,0.613464,0.0,0.349688,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.362174,0.000000
1,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.17451,0.205937,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.218413,0.000000,0.175856
2,0.000000,0.0,0.259957,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.081523,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.000000,...,0.320555,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.127306
4,0.137370,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.125606,...,0.000000,0.129197,0.118441,0.0,0.0,0.0,0.120625,0.000000,0.000000,0.000000


In [7]:
def assign_return_risk(row):
    # High Risk
    if row['Category'] in ['Dresses', 'Jumpsuits & Playsuits'] and (
        'sequin' in str(row['Name']).lower() or 'beaded' in str(row['Name']).lower() or 'lace' in str(row['Name']).lower() or 'bodycon' in str(row['Name']).lower() or 'plunge' in str(row['Name']).lower() or 'backless' in str(row['Name']).lower()
    ):
        return 'High'
    if row['Full Price ($)'] > 200:
        return 'High'

    # Medium Risk
    if row['Category'] in ['Tops', 'Bottoms', 'Knitwear']:
        return 'Medium'
    if row['Current Discount Percentage'] > 20:
        return 'Medium'

    # Low Risk
    if row['Category'] in ['Accessories', 'Shoes', 'Bags']:
        return 'Low'
    if 'cotton' in str(row['Care information']).lower():
        return 'Low'

    return 'Low'  # Default to low risk

# Apply the logic to the training and test data
train_df['return_risk'] = train_df.apply(assign_return_risk, axis=1)
test_df['return_risk'] = test_df.apply(assign_return_risk, axis=1)

# Display the distribution of return risk
print("Return Risk Distribution in Training Data:")
display(train_df['return_risk'].value_counts())

print("\nReturn Risk Distribution in Test Data:")
display(test_df['return_risk'].value_counts())

Return Risk Distribution in Training Data:


return_risk
Medium    24505
High      20634
Low       14861
Name: count, dtype: int64


Return Risk Distribution in Test Data:


return_risk
Low       22747
High      21101
Medium    16152
Name: count, dtype: int64

In [8]:
# Define the directory
output_directory = r'C:\Users\ADMIN\return_risk\data\stimulated_risk'

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the full file paths, including filenames
output_train_filepath = os.path.join(output_directory, 'stimulated_risk_train_data.csv')
output_test_filepath = os.path.join(output_directory, 'stimulated_risk_test_data.csv')

# Save the cleaned DataFrames to CSV files
train_df.to_csv(output_train_filepath, index=False)
test_df.to_csv(output_test_filepath, index=False)

print(f"Stimulated risk training data saved to: {output_train_filepath}")
print(f"Stimulated risk test data saved to: {output_test_filepath}")

Stimulated risk training data saved to: C:\Users\ADMIN\return_risk\data\stimulated_risk\stimulated_risk_train_data.csv
Stimulated risk test data saved to: C:\Users\ADMIN\return_risk\data\stimulated_risk\stimulated_risk_test_data.csv


In [9]:
# Define the features to use for training the model
categorical_features = ['Segment', 'Category', 'Color']
numerical_features = ['Full Price ($)', 'Current Discount Percentage']

# Create the feature set X by concatenating the categorical features, numerical features, and the X_train_text DataFrame
X = pd.concat([train_df[categorical_features], train_df[numerical_features], X_train_text], axis=1)

# Create the target variable y
y = train_df['return_risk']

# Display the shape of X and y
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (60000, 105)
Shape of y: (60000,)


In [10]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the new sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_val:", X_val.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_val:", y_val.shape)

Shape of X_train: (48000, 105)
Shape of X_val: (12000, 105)
Shape of y_train: (48000,)
Shape of y_val: (12000,)


## Train and Evaluate Logistic Regression

In [11]:
# Define the categorical and numerical features
categorical_features = ['Segment', 'Category', 'Color']
numerical_features = ['Full Price ($)', 'Current Discount Percentage']

# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

# Create the model pipeline
logreg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear'))
])

# Train the model
logreg_model.fit(X_train, y_train)

# Make predictions on the validation data
y_pred_logreg = logreg_model.predict(X_val)

# Evaluate the model's performance
print("Logistic Regression Model Evaluation")
print("Accuracy:", accuracy_score(y_val, y_pred_logreg))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_logreg))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_logreg))

C:\Users\ADMIN\return_risk\gtv\Lib\site-packages\sklearn\linear_model\_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Logistic Regression Model Evaluation
Accuracy: 0.8710833333333333

Classification Report:
              precision    recall  f1-score   support

        High       0.85      0.79      0.82      4137
         Low       0.86      0.92      0.89      2956
      Medium       0.89      0.91      0.90      4907

    accuracy                           0.87     12000
   macro avg       0.87      0.87      0.87     12000
weighted avg       0.87      0.87      0.87     12000


Confusion Matrix:
[[3266  379  492]
 [ 196 2706   54]
 [ 380   46 4481]]


Train and Evalutae Random Forest

In [17]:
from sklearn.ensemble import RandomForestClassifier

# Create the model pipeline with parameters to reduce overfitting
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, max_depth=20, min_samples_leaf=5))
])

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the validation data
y_pred_rf = rf_model.predict(X_val)

# Evaluate the model's performance
print("Random Forest Model Evaluation (with overfitting control)")
print("Accuracy:", accuracy_score(y_val, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))

Random Forest Model Evaluation (with overfitting control)
Accuracy: 0.9905833333333334

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.97      0.99      4137
         Low       0.98      1.00      0.99      2956
      Medium       0.99      1.00      0.99      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4027   61   49]
 [   1 2953    2]
 [   0    0 4907]]


In [18]:
# Evaluate the Random Forest model on the training data
y_pred_train_rf = rf_model.predict(X_train)

print("Random Forest Model - Training Set Performance")
print("Accuracy:", accuracy_score(y_train, y_pred_train_rf))
print("\nRandom Forest Model - Validation Set Performance")
print("Accuracy:", accuracy_score(y_val, y_pred_rf))

Random Forest Model - Training Set Performance
Accuracy: 0.9922708333333333

Random Forest Model - Validation Set Performance
Accuracy: 0.9905833333333334


In [20]:
# Create the folder if it doesn't exist
if not os.path.exists('trained_models'):
    os.makedirs('trained_models')

# Save the model to the folder
joblib.dump(rf_model, 'trained_models/tuned_random_forest_model.joblib')

print("Random Forest Model saved successfully!")

Random Forest Model saved successfully!


In [26]:
# Define the hyperparameter grid
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__bootstrap': [True, False]
}

# Create the RandomizedSearchCV object
rf_random = RandomizedSearchCV(estimator=rf_model, param_distributions=param_grid, n_iter=10, cv=3, verbose=2, random_state=42, n_jobs=-1)

# Fit the random search model
rf_random.fit(X_train, y_train)

# Print the best parameters
print("Best parameters found: ")
pprint(rf_random.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found: 
{'classifier__bootstrap': False,
 'classifier__max_depth': None,
 'classifier__min_samples_leaf': 2,
 'classifier__min_samples_split': 5,
 'classifier__n_estimators': 100}


In [27]:
# Get the best estimator from the random search
best_rf_model = rf_random.best_estimator_

# Make predictions on the validation data
y_pred_best_rf = best_rf_model.predict(X_val)

# Evaluate the model's performance
print("Tuned Random Forest Model Evaluation")
print("Accuracy:", accuracy_score(y_val, y_pred_best_rf))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_best_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_best_rf))

Tuned Random Forest Model Evaluation
Accuracy: 0.9924166666666666

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.98      0.99      4137
         Low       0.98      1.00      0.99      2956
      Medium       0.99      1.00      1.00      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4047   50   40]
 [   1 2955    0]
 [   0    0 4907]]


In [28]:
# Evaluate the tuned Random Forest model on the training data
y_pred_train_best_rf = best_rf_model.predict(X_train)

print("Tuned Random Forest Model - Training Set Performance")
print("Accuracy:", accuracy_score(y_train, y_pred_train_best_rf))
print("\nTuned Random Forest Model - Validation Set Performance")
print("Accuracy:", accuracy_score(y_val, y_pred_best_rf))

Tuned Random Forest Model - Training Set Performance
Accuracy: 0.9995208333333333

Tuned Random Forest Model - Validation Set Performance
Accuracy: 0.9924166666666666


In [29]:
# Save the model to the folder
joblib.dump(best_rf_model, 'trained_models/tuned_random_forest_model.joblib')

print("tuned_random_forest_model saved successfully!")

tuned_random_forest_model saved successfully!


## Train and Evaluate XGBoost

In [22]:
# Create a label encoder
le = LabelEncoder()

# Fit and transform the target variable
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# Create the model pipeline with parameters to reduce overfitting
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, max_depth=10, eta=0.1, subsample=0.8))
])

# Train the model
xgb_model.fit(X_train, y_train_encoded)

# Make predictions on the validation data
y_pred_xgb = xgb_model.predict(X_val)

# Evaluate the model's performance
print("XGBoost Model Evaluation (with overfitting control)")
print("Accuracy:", accuracy_score(y_val_encoded, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_val_encoded, y_pred_xgb, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val_encoded, y_pred_xgb))

XGBoost Model Evaluation (with overfitting control)
Accuracy: 0.992

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.98      0.99      4137
         Low       0.98      1.00      0.99      2956
      Medium       0.99      1.00      0.99      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4061   38   38]
 [   7 2948    1]
 [   3    9 4895]]


In [23]:
# Evaluate the XGBoost model on the training data
y_pred_train_xgb = xgb_model.predict(X_train)

print("XGBoost Model - Training Set Performance")
print("Accuracy:", accuracy_score(y_train_encoded, y_pred_train_xgb))
print("\nXGBoost Model - Validation Set Performance")
print("Accuracy:", accuracy_score(y_val_encoded, y_pred_xgb))

XGBoost Model - Training Set Performance
Accuracy: 0.9994791666666667

XGBoost Model - Validation Set Performance
Accuracy: 0.992


In [24]:
# Create the folder if it doesn't exist
if not os.path.exists('trained_models'):
    os.makedirs('trained_models')

# Save the model to the folder
joblib.dump(xgb_model, 'trained_models/tuned_xgboost_model.joblib')

print("XGBoost Model saved successfully!")

XGBoost Model saved successfully!


In [30]:
# Define the hyperparameter grid
param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7, 10],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__subsample': [0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9]
}

# Print the hyperparameter grid
print("XGBoost Hyperparameter Grid:")
pprint(param_grid_xgb)

XGBoost Hyperparameter Grid:
{'classifier__colsample_bytree': [0.7, 0.8, 0.9],
 'classifier__learning_rate': [0.05, 0.1, 0.2],
 'classifier__max_depth': [3, 5, 7, 10],
 'classifier__n_estimators': [100, 200, 300],
 'classifier__subsample': [0.7, 0.8, 0.9]}


In [31]:
# Create the RandomizedSearchCV object
xgb_random = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_grid_xgb, n_iter=10, cv=3, verbose=2, random_state=42, n_jobs=-1)

# Fit the random search model
xgb_random.fit(X_train, y_train_encoded)

# Print the best parameters
print("Best parameters found for XGBoost: ")
pprint(xgb_random.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found for XGBoost: 
{'classifier__colsample_bytree': 0.8,
 'classifier__learning_rate': 0.05,
 'classifier__max_depth': 10,
 'classifier__n_estimators': 100,
 'classifier__subsample': 0.9}


In [32]:
# Get the best estimator from the random search
best_xgb_model = xgb_random.best_estimator_

# Make predictions on the validation data
y_pred_best_xgb = best_xgb_model.predict(X_val)

# Evaluate the model's performance
print("Tuned XGBoost Model Evaluation")
print("Accuracy:", accuracy_score(y_val_encoded, y_pred_best_xgb))
print("\nClassification Report:")
print(classification_report(y_val_encoded, y_pred_best_xgb, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val_encoded, y_pred_best_xgb))

Tuned XGBoost Model Evaluation
Accuracy: 0.9920833333333333

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.98      0.99      4137
         Low       0.98      1.00      0.99      2956
      Medium       0.99      1.00      0.99      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4061   38   38]
 [   6 2950    0]
 [   2   11 4894]]


In [33]:
# Evaluate the tuned XGBoost model on the training data
y_pred_train_best_xgb = best_xgb_model.predict(X_train)

print("Tuned XGBoost Model - Training Set Performance")
print("Accuracy:", accuracy_score(y_train_encoded, y_pred_train_best_xgb))
print("\nTuned XGBoost Model - Validation Set Performance")
print("Accuracy:", accuracy_score(y_val_encoded, y_pred_best_xgb))

Tuned XGBoost Model - Training Set Performance
Accuracy: 0.9968125

Tuned XGBoost Model - Validation Set Performance
Accuracy: 0.9920833333333333


In [34]:
# Save the model to the folder
joblib.dump(best_xgb_model, 'trained_models/tuned_xgboost_model.joblib')

print("tuned_xgboost_model saved successfully!")

tuned_xgboost_model saved successfully!
